In [52]:
from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate , PromptTemplate
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma , FAISS
import os 
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

model = ChatGroq(model='Llama3-8b-8192' , groq_api_key = groq_api_key)

In [53]:
from langchain.schema import(
    AIMessage,
    HumanMessage,
    SystemMessage
)

speech = """
First and foremost, I offer my respectful salutations to the traditions of the Bharwad community, to all the revered saints and mahants, and to all those who have dedicated their lives to preserving this sacred tradition. Today, our joy has multiplied manifold. This time, the Maha Kumbh was not only historic but also a moment of great pride for us because, on this auspicious occasion, Mahant Shri Ram Bapu ji has been honoured with the title of Maha Mandaleshwar. This is a significant achievement and a moment of immense joy for all of us. My heartfelt congratulations to Ram Bapu ji and all the families of our community.

Over the past week, it felt as if the land of Bhavnagar had transformed into Lord Krishna’s Vrindavan, and to make this even more special, our revered brother’s Bhagavata Katha took place. The way devotion flowed, and the way people immersed themselves in Krishna’s love, created a truly divine atmosphere. My dear family, Bavaliyali Dham is not just a religious site; it is a symbol of faith, culture, and unity for the Bharwad community and many others.

By the grace of Naga Lakha Thakar, this sacred place has always provided the Bharwad community with true guidance and an immense legacy of noble inspiration. Today, the re-consecration of the Shri Naga Lakha Thakar Temple at this holy site has become a golden opportunity for us. Over the past week, there has been an atmosphere of grand celebration. The enthusiasm and excitement of the community are remarkable—I keep hearing praises from all around. In my heart, I feel that I should be there among you all, but due to my commitments in Parliament and work, it has been difficult to leave. However, when I hear about the magnificent ‘Raas’ (dance) performed by thousands of our sisters, I feel immense joy—they have truly brought Vrindavan to life right there!

The blending of faith, culture, and tradition is truly heart-warming and uplifting. Among all these events, I deeply appreciate the artists—brothers and sisters—who participated, making the occasion vibrant and delivering meaningful messages to society through their performances. I am confident that Bhai ji will continue to enlighten us with his wisdom through his storytelling. No matter how many times I express my gratitude, it will never be enough.

I sincerely thank Mahant Shri Ram Bapu ji and Bavaliyali Dham for allowing me to be a part of this sacred occasion. However, I must also seek forgiveness, as I could not be present with you all on this auspicious day. I know that you all have an equal right over me. But I assure you, whenever I visit that place in the future, I will definitely come to bow my head in reverence.
"""


In [54]:
speech

In [55]:
chat_message = [
    SystemMessage(content="you are and expert in expertise in summarizing the speeches"),
    HumanMessage(content=f"please provide a short and concise summary of following speech : \n Text: {speech}")
]

In [56]:
model.get_num_tokens(speech)

In [57]:
model(chat_message) # .content 

In [62]:
# prompt template text summarization 
from langchain.chains import LLMChain

generatetemplate = """
Write a summary of the following speech
speech : {speech}
Translate the precise summary to {language} language
"""


prompt = PromptTemplate(
    input_variables=['speech' , 'language'],
    template = generatetemplate
)

prompt


In [63]:
complete_prompt = prompt.format(speech = speech , language = "Japanese")
complete_prompt

In [64]:
model.get_num_tokens(speech)

In [69]:
llm_chain = LLMChain(llm = model , prompt = prompt)

summary = llm_chain.run({'speech':speech , 'language':'hindi'})
summary

# stuff document chain  text summarization 

In [73]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader(
    file_path=r"C:\Users\nex20\Documents\langchain\1-langchain\basics_steps\project2_midsem_report.pdf"
)
pages = loader.load_and_split()

pages

In [77]:
template = """ write a concise and short summary of the following Report
Report : {text}
"""

prompt = PromptTemplate(
    input_variables=['text'],
    template=template
)

In [79]:
from langchain.chains.summarize import load_summarize_chain # combine the splits and summarize each

chain = load_summarize_chain(model ,chain_type="stuff",prompt=prompt,verbose=True)
chain

In [81]:
output_summary = chain.run(pages)

output_summary

## mapreduce to summarize large documents 

In [84]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader(
    file_path=r"C:\Users\nex20\Documents\langchain\1-langchain\basics_steps\project2_midsem_report.pdf"
)
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000 , 
    chunk_overlap = 100
)

final_documents = text_splitter.split_documents(pages)
len(final_documents)

In [ ]:
template = """ write a concise and short summary of the following Report
Report : {text}
"""

prompt = PromptTemplate(
    input_variables=['text'],
    template=template
)

final_template = """
provide the final summary of entire speech with these important points
add the title , start the precise summary with introduction anf provide summary in number of points for the speech 
Speech : {text}
"""

final_prompt_template = PromptTemplate(
    input_variables=['text'],
    template=final_template
)


from langchain.chains.summarize import load_summarize_chain # combine the splits and summarize each

chain = load_summarize_chain(model ,
                            chain_type="map_reduce",
                            map_prompt=prompt, # summary of smaller chunks
                            combine_prompt = final_prompt_template, # combine with this and give whole result
                            verbose=True)


output = chain.run(final_documents)
output

# refine chain summarization

In [89]:
chain = load_summarize_chain(
    llm = model,
    chain_type='refine',
    verbose=True
)


output_summary = chain.run(final_documents)
output_summary